# 02_silver_notebook_crm_account

**Purpose**: Build unified account dimension from CRM Bronze + Legacy CIS static data.

**Source**:
- `APAC_CRM_Analytics_LH.src_crm_account` (Bronze — active CRM accounts)
- `APAC_CRM_Analytics_LH.src_crm_legacy_cis` (Bronze — static legacy data)

**Output**: `APAC_Reporting_LH.clean_crm_account` (Silver)

**Logic**:
- Bronze accounts: cast + passthrough
- Legacy: deduplicated accounts not already in Bronze (matched on GCID or AccountName)
- Legacy keys: `LEGACY_` + ROW_NUMBER (stable — legacy data is static)

In [ ]:
# =============================================================================
# Cell 1: Setup & Configuration
# =============================================================================
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window

BRONZE_LH = "APAC_CRM_Analytics_LH"
TARGET_TABLE = "APAC_Reporting_LH.clean_crm_account"

SRC_ACCOUNT    = f"{BRONZE_LH}.src_crm_account"
SRC_LEGACY_CIS = f"{BRONZE_LH}.src_crm_legacy_cis"

In [ ]:
# =============================================================================
# Cell 2: Load Bronze + Schema Check
# =============================================================================
df_account = spark.sql(f"SELECT * FROM {SRC_ACCOUNT}")
df_legacy  = spark.sql(f"SELECT * FROM {SRC_LEGACY_CIS}")

print("=== src_crm_account ===")
df_account.printSchema()
display(df_account.limit(3))
print(f"Rows: {df_account.count()}")

print("\n=== src_crm_legacy_cis ===")
df_legacy.printSchema()
display(df_legacy.limit(3))
print(f"Rows: {df_legacy.count()}")

In [ ]:
# =============================================================================
# Cell 3: Bronze Accounts — cast to standard types
# =============================================================================
df_bronze = df_account.select(
    F.col("AccountKey").cast(StringType()).alias("AccountKey"),
    F.col("GCID").cast(StringType()).alias("GCID"),
    F.col("AccountName").cast(StringType()).alias("AccountName"),
    F.col("ParentAccountName").cast(StringType()).alias("ParentAccountName"),
    F.col("GlobalParentAccountName").cast(StringType()).alias("GlobalParentAccountName"),
    F.col("DunsNumber").cast(StringType()).alias("DunsNumber"),
    F.col("Tiers").cast(StringType()).alias("Tiers"),
    F.col("Industry").cast(StringType()).alias("Industry"),
    F.col("PrimarySICCode").cast(StringType()).alias("PrimarySICCode"),
    F.col("Country").cast(StringType()).alias("Country"),
    F.col("CreateDate").cast(DateType()).alias("CreateDate"),
    F.col("ModifiedDate").cast(DateType()).alias("ModifiedDate"),
)

print(f"Bronze accounts: {df_bronze.count()}")

In [ ]:
# =============================================================================
# Cell 4: Legacy Accounts — deduplicate, exclude if already in Bronze
# =============================================================================

# Step 1: deduplicate legacy on natural key
df_legacy_dedup = df_legacy.select(
    F.col("Account").cast(StringType()).alias("Account"),
    F.col("GCID").cast(StringType()).alias("GCID"),
    F.col("Tiers").cast(StringType()).alias("Tiers"),
    F.col("`Industry _Account Name_ _Account_`").cast(StringType()).alias("Industry"),
).distinct()

# Step 2: collect Bronze GCID and AccountName sets for exclusion
bronze_gcids   = {r.GCID for r in df_bronze.select("GCID").where(F.col("GCID").isNotNull()).collect()}
bronze_names   = {r.AccountName.upper().strip() for r in df_bronze.select("AccountName").where(F.col("AccountName").isNotNull()).collect()}

# Step 3: filter legacy to accounts not already in Bronze
def not_in_bronze(gcid, account):
    if gcid is not None and gcid in bronze_gcids:
        return False
    if account is not None and account.upper().strip() in bronze_names:
        return False
    return True

from pyspark.sql.functions import udf
from pyspark.sql.types import BooleanType

not_in_bronze_udf = udf(not_in_bronze, BooleanType())

df_legacy_new = df_legacy_dedup.filter(
    not_in_bronze_udf(F.col("GCID"), F.col("Account"))
)

# Step 4: assign stable LEGACY_ keys using ROW_NUMBER (data is static)
window_legacy = Window.orderBy("Account", "GCID")

df_legacy_final = df_legacy_new.select(
    F.concat(F.lit("LEGACY_"), F.row_number().over(window_legacy).cast(StringType())).alias("AccountKey"),
    F.col("GCID").alias("GCID"),
    F.col("Account").alias("AccountName"),
    F.lit(None).cast(StringType()).alias("ParentAccountName"),
    F.lit(None).cast(StringType()).alias("GlobalParentAccountName"),
    F.lit(None).cast(StringType()).alias("DunsNumber"),
    F.col("Tiers").alias("Tiers"),
    F.col("Industry").alias("Industry"),
    F.lit(None).cast(StringType()).alias("PrimarySICCode"),
    F.lit(None).cast(StringType()).alias("Country"),
    F.lit(None).cast(DateType()).alias("CreateDate"),
    F.lit(None).cast(DateType()).alias("ModifiedDate"),
)

print(f"Legacy accounts (new only): {df_legacy_final.count()}")

In [ ]:
# =============================================================================
# Cell 5: Union Bronze + Legacy
# =============================================================================
df_final = df_bronze.unionByName(df_legacy_final)

print(f"Bronze: {df_bronze.count()} | Legacy new: {df_legacy_final.count()} | Total: {df_final.count()}")
df_final.printSchema()
display(df_final.limit(5))

In [ ]:
# =============================================================================
# Cell 6: Write to Silver
# =============================================================================
print(f"Writing to {TARGET_TABLE}...")
df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(TARGET_TABLE)

df_check = spark.sql(f"SELECT * FROM {TARGET_TABLE}")
print(f"Success. Rows written: {df_check.count()}")
print(f"Columns: {len(df_check.columns)}")
display(df_check.limit(5))